In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:120b-cloud")
print('Model is ready')

In [ ]:
from typing import TypedDict

class ReflectionState(TypedDict):
    task: str
    draft: str
    critique: str
    iteration: int

print('Reflection State is ready')

In [ ]:
# Generator node
def generate(state: ReflectionState):

    if state["draft"]:
        prompt = f"""
        You are an expert technical writer.

        Original task:
        {state["task"]}

        Previous draft:
        {state["draft"]}

        Critic feedback:
        {state["critique"]}

        Improve the previous draft based on the critic's feedback.

        Return only the improved draft.
        """
    else:
        prompt = f"""
        You are an expert technical writer.

        Write a high-quality answer for the following task:

        {state["task"]}

        Return only the draft.
        """

    response = model.invoke(prompt)

    return {
        "draft": response.content,
        "iteration": state["iteration"] + 1
    }

print('Generate node is ready')

In [ ]:
# Reflection/Critic node
def reflect(state: ReflectionState):

    prompt = f"""
    You are a strict reviewer.

    Review the following answer against the original task.

    Original task:
    {state["task"]}

    Draft:
    {state["draft"]}

    Analyze:

    1. Accuracy
    2. Completeness
    3. Clarity
    4. Relevance
    5. Technical quality

    If the answer is good enough, start your response with:

    PASS

    Otherwise, start your response with:

    REVISE

    Then provide detailed feedback.
    """

    response = model.invoke(prompt)

    return {
        "critique": response.content
    }

print('Reflection node is ready')

In [ ]:
# Should continue node
def should_continue(state: ReflectionState):

    critique = state["critique"]

    if critique.strip().upper().startswith("PASS"):
        return "accept"

    if state["iteration"] >= 3:
        return "accept"

    return "improve"

print('should continue is ready')

In [ ]:
from langgraph.graph import StateGraph, START, END


builder = StateGraph(ReflectionState)

builder.add_node("generate", generate)
builder.add_node("reflect", reflect)

builder.add_edge(START, "generate")

builder.add_edge(
    "generate",
    "reflect"
)

builder.add_conditional_edges(
    "reflect",
    should_continue,
    {
        "improve": "generate",
        "accept": END
    }
)

graph = builder.compile()
print('Graph is ready')
graph

In [11]:
result = graph.invoke({
    "task": """
    Please suggest few tourist places in New Delhi for 2 days tour.
    """,
    "draft": "",
    "critique": "",
    "iteration": 0
})
result

{'task': '\n    Please suggest few tourist places in New Delhi for 2 days tour.\n    ',
 'draft': '**Two‑Day Tourist Itinerary for New\u202fDelhi**\n\n---\n\n### Day\u202f1 – Old Delhi & Central Highlights  \n\n| Time | Spot | Why Visit / Highlights | Practical Tips |\n|------|------|------------------------|----------------|\n| **08:00\u202f–\u202f09:00** | **Breakfast at *Karim’s* (Jama Masjid)** | Legendary Mughlai fare – parathas, kebabs, jalebi. | Arrive early to avoid queues; cash preferred. |\n| **09:30\u202f–\u202f11:30** | **Red\u202fFort (Lal\u202fQila)** | UNESCO World Heritage site; impressive sandstone walls, Diwan‑i‑Khas,\u202fMumtaz\u202fMasjid. | Allocate ~1.5\u202fh; rent an audio guide (₹150). |\n| **11:45\u202f–\u202f12:30** | **Jama Masjid** | One of India’s largest mosques; panoramic city view from the minaret. | Dress modestly; remove shoes. |\n| **12:45\u202f–\u202f13:45** | **Chandni Chowk Walk** | Bustling market lanes; spice‑laden bazaars, Chandni Chowk’s hist